# RG3 가스 신호 결합: 중첩 검증
전체 가스17개 유지. 테스트 라벨을 사용하지 않는 학습 점수 결합·판정기준 보정.

In [1]:
from pathlib import Path
import json,subprocess
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler,StandardScaler
from sklearn.linear_model import Ridge,LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'
tables=root/'reports/tables'
d=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d.loc[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.str.contains('가스',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);groups=d.comparison_block
assert len(d)==583 and y.sum()==17
blocks=groups.drop_duplicates().tolist()
stdcols=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
relations={
'pressure_residual':('Max_Injection_Pressure',['Max_Injection_Speed','Filling_Time']),
'plasticizing_residual':('Plasticizing_Time',['Average_Screw_RPM','Average_Back_Pressure']),
'cushion_residual':('Cushion_Position',['Plasticizing_Position','Max_Injection_Pressure']),
'mold_residual':('Mold_Temperature_4',['Mold_Temperature_3','Barrel_Temperature_5'])}
# Original four-residual setup preserved for an exact comparison.
# Average RPM remains definition-uncertain; no physical interpretation of its residual.
cache={}
def res_features(ref,q):
    key=(tuple(ref),tuple(q))
    if key in cache:return cache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    normal=d.iloc[ref].loc[y[ref]==0];query=d.iloc[q]
    out=np.empty((len(q),4))
    for j,(target,predictors) in enumerate(relations.values()):
        sc=RobustScaler().fit(normal[predictors])
        m=Ridge(alpha=10).fit(sc.transform(normal[predictors]),normal[target])
        pred=m.predict(sc.transform(normal[predictors]))
        scale=max(np.sqrt(np.mean((normal[target].to_numpy()-pred)**2)),float(normal[target].std(ddof=0))*.1,1e-6)
        out[:,j]=(query[target].to_numpy()-m.predict(sc.transform(query[predictors])))/scale
    cache[key]=out
    return out
def rf_fit(x,yy,q):
    if len(np.unique(yy))<2:return np.full(len(q),float(yy[0]) if len(yy) else 0.)
    m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1))
    return m.fit(x,yy).predict_proba(q)[:,1]
paircache={}
def pair_predict(train,test):
    key=(tuple(train),tuple(test))
    if key in paircache:return paircache[key]
    a=rf_fit(d.iloc[train][stdcols],y[train],d.iloc[test][stdcols])
    if len(np.unique(y[train]))<2:
        b=np.full(len(test),float(y[train][0]) if len(train) else 0.)
    else:
        tr_res=np.full((len(train),4),np.nan)
        for block in groups.iloc[train].unique():
            mask=groups.iloc[train].eq(block).to_numpy()
            ref=train[~mask];query=train[mask]
            if len(ref) and (y[ref]==0).any():tr_res[mask]=res_features(ref,query)
        b=rf_fit(tr_res,y[train],res_features(train,test))
    paircache[key]=np.column_stack([a,b])
    return paircache[key]
scores=[];thresholds=[];foldlogs=[]
names=['std4','residual4','equal_mean','max_union','stack_logit2']
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        test=np.flatnonzero(groups.eq(block))
        train=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[train]))<2:continue
        outer=pair_predict(train,test)
        inner=np.full((len(train),2),np.nan)
        for innerblock in groups.iloc[train].unique():
            mask=groups.iloc[train].eq(innerblock).to_numpy()
            ref=train[~mask];query=train[mask]
            if len(ref):inner[mask]=pair_predict(ref,query)
        valid=np.isfinite(inner).all(axis=1)
        # Inner OOF scores only; no outer labels or fitted-on-self RF scores.
        meta=make_pipeline(StandardScaler(),LogisticRegression(C=1,class_weight='balanced',random_state=42,max_iter=1000))
        meta.fit(inner[valid],y[train][valid])
        outer_scores={'std4':outer[:,0],'residual4':outer[:,1],'equal_mean':outer.mean(axis=1),'max_union':outer.max(axis=1),'stack_logit2':meta.predict_proba(outer)[:,1]}
        inner_scores={'std4':inner[:,0],'residual4':inner[:,1],'equal_mean':inner.mean(axis=1),'max_union':inner.max(axis=1),'stack_logit2':meta.predict_proba(inner[valid])[:,1]}
        for name in names:
            cal=inner_scores[name] if name!='stack_logit2' else np.full(len(train),np.nan)
            if name=='stack_logit2':
                # Meta threshold uses leave-one-inner-block-out meta predictions as well.
                for ib in groups.iloc[train].unique():
                    mask=groups.iloc[train].eq(ib).to_numpy()&valid;ref=valid&~mask
                    if len(np.unique(y[train][ref]))<2:continue
                    mm=make_pipeline(StandardScaler(),LogisticRegression(C=1,class_weight='balanced',random_state=42,max_iter=1000))
                    mm.fit(inner[ref],y[train][ref]);cal[mask]=mm.predict_proba(inner[mask])[:,1]
            normal_cal=cal[(y[train]==0)&np.isfinite(cal)]
            threshold=float(np.quantile(normal_cal,.99,method='higher')) if len(normal_cal) else 1.
            thresholds.append(dict(scheme=scheme,block=block,model=name,threshold=threshold,normal_calibration_n=len(normal_cal),normal_calibration_above=int((normal_cal>threshold).sum())))
            for k,i in enumerate(test):
                scores.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],comparison_block=block,actual=int(y[i]),score=float(outer_scores[name][k]),threshold=threshold,predicted_fixed05=bool(outer_scores[name][k]>=.5),predicted_train_fpr1=bool(outer_scores[name][k]>threshold)))
        foldlogs.append(dict(scheme=scheme,block=block,train=len(train),train_gas=int(y[train].sum()),test=len(test),test_gas=int(y[test].sum())))
        print('Completed',scheme,block,flush=True)
pred=pd.DataFrame(scores);results=[]
for (scheme,name),g in pred.groupby(['scheme','model'],sort=False):
    selected=np.zeros(len(g),bool);gg=g.reset_index(drop=True)
    for block,part in gg.groupby('comparison_block',sort=False):
        ids=part.index.to_numpy();tie=np.random.RandomState(42).rand(len(ids))
        selected[ids[np.lexsort((tie,-part.score.to_numpy()))[:int(np.ceil(len(ids)*.1))]]]=True
    for rule,col in [('fixed05','predicted_fixed05'),('train_fpr1','predicted_train_fpr1')]:
        tn,fp,fn,tp=confusion_matrix(g.actual,g[col],labels=[0,1]).ravel()
        results.append(dict(scheme=scheme,model=name,rule=rule,shots=len(g),gas=int(g.actual.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(g.actual,g.score),top10_TP=int(gg.actual.to_numpy()[selected].sum()),top10_selected=int(selected.sum())))
metrics=pd.DataFrame(results)
pred.to_csv(tables/'rg3_nested_ensemble_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_nested_ensemble_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(thresholds).to_csv(tables/'rg3_nested_ensemble_thresholds.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(foldlogs).to_csv(tables/'rg3_nested_ensemble_folds.csv',index=False,encoding='utf-8-sig')
old=pd.read_csv(tables/'rg3_relational_feature_oof_predictions.csv',parse_dates=['TimeStamp'])
for name,setting in [('std4','previous_std10'),('residual4','normal_relation_residual4')]:
    p=pred.loc[pred.scheme.eq('leave_one_block_out')&pred.model.eq(name)]
    chk=p.merge(old.loc[old.setting.eq(setting),['TimeStamp','probability']],on='TimeStamp',validate='one_to_one')
    assert len(chk)==583 and np.allclose(chk.score,chk.probability)
lines=['# RG3 가스: 서로 다른 탐지 신호의 중첩 결합','',
'## 조건','',
'- RH 가스17 + 정상566 전체 유지. 미성형 제외. 같은 RF200/depth8/leaf3/balanced_subsample/seed42.',
'- 흔들림 RF와 기존 정상관계잔차 RF를 정확히 재현. 잔차4개 중 평균RPM 관련 식은 정의 불확실성이 있어 물리 원인으로 해석하지 않음.',
'- 결합 후보는 사전 고정 평균, 최대값(OR에 대응), 두점수 로지스틱(C1,balanced)뿐. 단독모델과 같이 보고.',
'- 바깥 구간을 완전히 제외하고, 결합 학습 점수는 안쪽 구간제외 OOF로 생성. 잔차의 정상 회귀도 각 RF 학습 구간 안에서 다시 학습.',
'- fixed05는 고정0.5. train_fpr1은 학습 정상의 안쪽 예측 점수99% 분위수보다 큰 점수. 평가 정상 오탐1%를 보장하거나 정확히6개로 강제한 것이 아님.',
'- 로지스틱 기준 보정은 안쪽 구간을 하나 더 빼고 결합모델 예측을 만들어 사용. 모든 모델선택·분할 방식은 반복 탐색의 영향을 받아 독립 최종 평가로 주장하지 않음.',
'- 과거구간만 학습 결과는 후반440개/가스13개로 별도 표기. 안쪽 결합학습의 구간분할은 과거 학습집합 내부 교차검증이며 외부 미래구간은 사용하지 않음.',
'- 상위10%는 각 평가구간별 배치 검사 예산. 실시간 임계값 성능과 별개.',
'','## 결과','',
'| scheme | model | rule | shots | gas | TP | FP | FN | TN | AP | top10_TP | top10_selected |',
'|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.rule} | {r.shots} | {r.gas} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.AP:.4f} | {r.top10_TP} | {r.top10_selected} |')
lines+=['','새결합이 TP를 올려도 FP가 증가하면 동일오탐에서 개선이라고 하지 않는다. AP와판정성능을 분리해서 해석하며, 학습기준보정으로 외부오탐을 통제할 수 있는지도 함께 점검했다. 새불량 발견 사례의 자세한 점수는 rg3_nested_ensemble_predictions.csv에 저장.']
(root/'reports/rg3_nested_ensemble_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(metrics.to_string(index=False))


Completed leave_one_block_out 20201021_001626_RG3
Completed leave_one_block_out 20201022_002405_RG3
Completed leave_one_block_out 20201022_044026_RG3
Completed leave_one_block_out 20201023_010128_RG3
Completed leave_one_block_out 20201023_044148_RG3
Completed past_blocks_only 20201022_044026_RG3
Completed past_blocks_only 20201023_010128_RG3
Completed past_blocks_only 20201023_044148_RG3
             scheme        model       rule  shots  gas  TP  FP  FN  TN       AP  top10_TP  top10_selected
leave_one_block_out         std4    fixed05    583   17   1   6  16 560 0.138711         6              62
leave_one_block_out         std4 train_fpr1    583   17   3  13  14 553 0.138711         6              62
leave_one_block_out    residual4    fixed05    583   17   1   1  16 565 0.084763         2              62
leave_one_block_out    residual4 train_fpr1    583   17   1   0  16 566 0.084763         2              62
leave_one_block_out   equal_mean    fixed05    583   17   0   0  17 566 0.